# Calculate the percentage of anomaly
#### Jose Valles (jose.valles.leon@gmail.com)

### Importing Libraries

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('classic')
import numpy as np
import calendar

sns.set()

from IPython.display import HTML

Import basin level code 2 and 3

In [2]:
BASIN_LEVEL3 = pd.read_csv(f'../waterbalance/balance_hidrico_regional/output_modelo/cuenca_nivel3.csv',index_col="Codigo")
BASIN_LEVEL2 = pd.read_csv(f'../waterbalance/balance_hidrico_regional/output_modelo/cuenca_nivel2.csv',index_col="Codigo")

In [3]:
def importmodelvariable(model_variable):
    df = pd.read_csv(f'../waterbalance/balance_hidrico_regional/output_modelo/{model_variable}.csv')
    df = df.rename(columns={'-1': 'year','-1.1':'month'})
    df['date'] = pd.to_datetime(dict(year=df['year'],month=df['month'],day=1))
    df = df.set_index('date')
    df['days_in_month'] = df.index.days_in_month
    return df

def convertRunoff2Discharge(df_runoff):
    df_runoff_selected = df_runoff.drop(['year','month','days_in_month'],axis=1)
    df_discharge = pd.DataFrame(df_runoff_selected.values*1000*BASIN_LEVEL3.values,columns=df_runoff_selected.columns)
    df_discharge['days_in_month'] = df_runoff['days_in_month'].values
    df_discharge = df_discharge.loc[:, df_discharge.columns != 'days_in_month'].divide(df_discharge["days_in_month"]*24*3600, axis="index")
    df_discharge['date'] = df_runoff.index.values
    df_discharge = df_discharge.set_index('date')
    df_discharge['year'] = df_runoff['year'].values
    df_discharge['month'] = df_runoff['month'].values
    return df_discharge

def defineHydroSOScategory(VARIABLE_MENSUAL,VARIABLE_AVERAGE,VARIABLE):
    # create empty columns in the dataframe
    VARIABLE_MENSUAL['mean'] = np.nan
    VARIABLE_MENSUAL['average_percentage'] = np.nan
    VARIABLE_MENSUAL['rank_average'] = np.nan
    VARIABLE_MENSUAL['non_missing'] = np.nan


    for i in range(len(VARIABLE_MENSUAL)):
        # Extract the current month 
        m = VARIABLE_MENSUAL.month[i]
        # Extract the current year
        y = VARIABLE_MENSUAL.year[i]
        VARIABLE_MENSUAL.loc[VARIABLE_MENSUAL.eval('month==@m & year==@y'),'rank_average']  = VARIABLE_MENSUAL.query('month==@m')[VARIABLE].rank()
        VARIABLE_MENSUAL.loc[VARIABLE_MENSUAL.eval('month==@m & year==@y'),'non_missing']  = VARIABLE_MENSUAL.query('month==@m')[VARIABLE].notnull().sum()
        VARIABLE_MENSUAL.loc[VARIABLE_MENSUAL.eval('month==@m & year==@y'),'mean'] = VARIABLE_AVERAGE.query('month == @m')[VARIABLE].item()
        VARIABLE_MENSUAL.loc[VARIABLE_MENSUAL.eval('month==@m & year==@y'),'average_percentage'] = (VARIABLE_MENSUAL[VARIABLE][i] - VARIABLE_AVERAGE.query('month == @m')[VARIABLE].item()) / VARIABLE_AVERAGE.query('month == @m')[VARIABLE].item()

    VARIABLE_MENSUAL['percentile'] = VARIABLE_MENSUAL['rank_average']/(VARIABLE_MENSUAL['non_missing']+1)
    criteria = [VARIABLE_MENSUAL['percentile'].between(0.90,1.00),
            VARIABLE_MENSUAL['percentile'].between(0.75,0.90),
            VARIABLE_MENSUAL['percentile'].between(0.25,0.75),
            VARIABLE_MENSUAL['percentile'].between(0.10,0.25),
            VARIABLE_MENSUAL['percentile'].between(0.00,0.10)]

    values = ['High flow','Above normal','Normal range','Below normal','Low flow']

    VARIABLE_MENSUAL['percentile_range'] = np.select(criteria,values,None)
    return VARIABLE_MENSUAL

In [4]:
hydrological_variable = ['Escorrentia_total','Escorrentia_sup','Escorrentia_sub','Pmedias','ETR','HumedadSuelo']

for hydro in hydrological_variable:
    if hydro == "Escorrentia_total":
        RUNOFF_total = importmodelvariable(hydro)
    elif hydro == "Pmedias":
        PRECIP = importmodelvariable(hydro)
    elif hydro == "ETR":
        ETR = importmodelvariable(hydro)
    elif hydro == "HumedadSuelo":
        SM = importmodelvariable(hydro)
    elif hydro == 'Escorrentia_sup':
        RUNOFF_sup = importmodelvariable(hydro)
    elif hydro == 'Escorrentia_sub':
        RUNOFF_sub = importmodelvariable(hydro)

In [5]:
HTML(PRECIP.tail(6).to_html(index=False))

year,month,101,102,103,105,106,107,108,109,110,111,112,114,115,116,117,119,120,123,125,128,130,131,132,133,134,135,136,137,138,139,140,142,146,148,150,155,158,160,163,165,167,168,170,171,172,173,174,175,176,177,178,179,180,183,186,189,190,193,196,199,201,204,208,210,211,212,213,214,215,216,217,220,221,222,223,224,225,226,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,260,262,264,266,268,270,274,275,276,277,279,280,281,282,283,284,285,286,287,288,289,290,293,294,295,297,298,300,301,305,306,310,312,315,316,318,320,325,330,333,334,335,338,406,407,410,411,412,413,414,415,416,417,418,419,420,422,424,426,428,430,431,432,433,434,435,436,437,438,440,441,442,443,444,445,446,447,448,449,450,452,453,454,455,457,458,459,501,502,503,504,505,506,507,508,509,510,511,512,513,514,515,516,517,518,519,520,522,524,526,530,532,534,536,538,540,541,542,543,544,545,546,548,549,550,551,552,553,554,555,556,557,558,559,560,561,562,563,564,565,566,567,568,569,570,571,575,580,581,582,583,584,585,586,587,588,589,600,601,602,603,604,605,606,607,608,609,610,611,612,613,615,616,618,620,625,630,635,640,645,650,651,652,653,654,655,656,657,658,660,661,662,666,670,673,677,680,683,687,days_in_month
2025,9,191.160,190.650,206.410,183.570,181.260,189.800,165.860,182.220,186.960,187.090,183.580,182.650,172.940,166.380,168.17,147.890,135.410,128.430,133.880,139.350,164.310,163.800,141.220,135.960,127.950,114.910,119.030,117.340,101.930,82.704,87.283,76.071,82.670,64.946,95.718,76.679,75.170,58.498,63.463,56.900,53.478,40.469,60.495,47.600,33.415,30.820,58.558,43.621,27.359,24.292,29.756,30.782,20.645,18.641,26.273,35.283,52.795,72.568,66.675,70.413,79.591,79.013,80.443,83.794,83.619,83.386,82.250,81.725,82.762,82.344,81.144,78.365,78.413,75.795,83.117,82.603,82.751,83.020,72.520,73.026,70.838,70.175,69.924,75.793,78.180,81.052,78.210,81.437,78.834,75.668,70.899,62.496,55.485,55.737,48.946,47.237,47.5180,43.6400,42.1030,41.7240,42.8880,45.8040,44.9340,46.504,45.9210,45.9520,46.083,46.1200,40.789,44.594,44.282,42.538,39.554,47.0160,45.195,39.984,42.014,45.5030,41.274,38.834,30.980,33.738,28.630,29.107,28.439,27.400,29.869,28.186,27.179,25.491,23.643,22.923,21.826,22.769,23.295,28.214,28.852,30.184,33.302,33.959,124.360,74.195,121.950,122.450,122.470,123.350,123.970,110.980,108.080,107.240,97.129,93.571,64.581,63.878,73.612,75.319,57.782,56.448,60.491,50.662,49.784,49.337,54.150,46.460,57.983,43.197,41.267,36.240,38.469,34.159,31.175,26.836,39.414,41.925,46.683,70.435,42.916,38.067,35.947,27.466,35.079,34.074,34.112,33.915,190.960,193.050,183.120,154.030,132.480,116.350,91.004,70.773,60.157,179.130,180.710,189.220,179.920,165.800,141.740,98.779,153.920,117.950,123.690,109.560,93.806,106.670,94.252,98.153,92.115,93.741,157.510,135.720,62.690,69.645,65.116,58.388,55.973,58.507,57.135,60.761,55.290,57.796,63.476,55.674,62.936,52.616,49.329,52.991,53.786,59.792,50.271,47.806,41.498,51.084,39.533,38.152,42.106,57.315,52.422,58.594,69.549,64.165,72.305,54.619,45.632,40.943,33.698,51.740,46.812,32.726,36.889,46.191,56.629,46.383,35.284,35.755,39.767,45.347,45.171,42.971,45.826,45.941,47.870,48.230,36.811,40.433,40.816,44.347,47.722,48.023,49.188,47.882,49.143,47.5970,48.1310,49.0730,48.4480,58.734,57.853,56.676,60.069,56.803,52.861,51.284,48.6060,49.008,48.4760,49.2600,48.3180,49.304,47.9250,48.5510,45.5900,47.9500,47.1650,45.1910,30
2025,10,76.737,90.399,100.930,117.890,119.930,108.850,106.370,129.790,133.460,134.660,135.560,135.120,135.750,135.940,134.44,125.450,92.691,93.956,97.798,106.810,77.662,82.649,88.250,88.932,92.953,97.493,96.255,97.643,113.920,121.270,128.890,127.720,120.460,112.940,108.210,105.280,102.910,95.779,101.280,107.200,115.320,123.790,152.140,162.450,157.280,151.360,146.080,129.260,144.560,142.460,130.850,130.610,140.620,144.030,139.110,101.060,87.167,95.142,97.443,101.930,109.270,108.730,113.600,102.380,105.050,106.140,105.660,107.280,105.550,106.320,110.880,108.400,103.340,100.440,103.190,103.240,103.470,103.970,94.702,95.268

### Convert from runoff to discharge

In [6]:
DISCHARGE = convertRunoff2Discharge(RUNOFF_total)

In [7]:
HTML(DISCHARGE.tail(6).to_html(index=True))

,101,102,103,105,106,107,108,109,110,111,112,114,115,116,117,119,120,123,125,128,130,131,132,133,134,135,136,137,138,139,140,142,146,148,150,155,158,160,163,165,167,168,170,171,172,173,174,175,176,177,178,179,180,183,186,189,190,193,196,199,201,204,208,210,211,212,213,214,215,216,217,220,221,222,223,224,225,226,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,260,262,264,266,268,270,274,275,276,277,279,280,281,282,283,284,285,286,287,288,289,290,293,294,295,297,298,300,301,305,306,310,312,315,316,318,320,325,330,333,334,335,338,406,407,410,411,412,413,414,415,416,417,418,419,420,422,424,426,428,430,431,432,433,434,435,436,437,438,440,441,442,443,444,445,446,447,448,449,450,452,453,454,455,457,458,459,501,502,503,504,505,506,507,508,509,510,511,512,513,514,515,516,517,518,519,520,522,524,526,530,532,534,536,538,540,541,542,543,544,545,546,548,549,550,551,552,553,554,555,556,557,558,559,560,561,562,563,564,565,566,567,568,569,570,571,575,580,581,582,583,584,585,586,587,588,589,600,601,602,603,604,605,606,607,608,609,610,611,612,613,615,616,618,620,625,630,635,640,645,650,651,652,653,654,655,656,657,658,660,661,662,666,670,673,677,680,683,687,year,month
date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2025-09-01,35.414815,41.607948,37.406782,6.971424,16.035259,60.251929,80.226921,40.778095,2.310606,17.853326,3.056944,9.171632,7.512341,1.889654,10.203606,34.417128,10.347344,10.791546,21.109738,10.440275,7.590225,32.500125,13.148261,32.870947,13.471811,82.533854,17.682177,18.212375,14.293790,4.084066,0.161150,3.451130,16.754250,3.981502,31.497815,10.964636,13.663963,0.957218,1.762833,2.963318,9.626696,3.477120,20.102002,21.665346,3.271600,2.340543,19.635000,6.478622,2.489969,3.962527,7.091535,4.606886,0.490569,4.991641,21.247963,5.524293,0.125260,37.875520,3.096504,2.684354,0.110076,6.195572,12.110807,0.954691,2.167148,1.167299,0.999148,1.120247,2.044179,2.316066,7.021886,0.074283,1.946955,1.377649,1.808184,0.157747,0.466972,3.206868,0.693093,0.212404,1.297889,0.782571,2.261984,3.576593,0.545978,6.636959,1.020542,0.515234,0.164051,0.477532,3.217354,3.518372,1.864742,0.538247,0.520971,0.275964,0.324440,0.240349,0.283832,0.672668,1.000406,4.065826,0.082392,3.318389,0.330178,0.105672,0.068774,0.156253,1.077765,0.393781,0.832294,0.647419,2.213130,0.789950,0.138281,0.183667,0.342387,0.655722,0.153507,0.439532,0.186856,4.484886,0.087916,0.242546,0.040760,5.748275,3.182920,0.812938,0.147965,4.609979,3.464489,0.224536,7.305076,0.564950,7.897762,0.567233,2.757535,9.337967,0.322148,0.210624,42.792389,0.967156,5.601987,5.727802,7.204874,15.886381,20.818625,1.137799,5.578029,14.438497,3.680723,12.467481,0.159174,1.903136,1.567638,5.874954,1.689782,1.786961,1.845576,0.606235,1.861830,6.086395,7.070807,11.627006,11.432796,1.191325,1.473736,2.666685,4.447429,6.487737,10.812309,20.886831,7.964160,6.420940,0.683333,25.264856,1.119745,2.931257,1.328361,6.208757,3.305947,0.181605,0.007076,0.938047,14.063920,46.040664,26.067191,59.441811,24.393320,25.092369,19.689778,14.007821,26.631383,5.144028,4.725926,15.825297,22.673032,12.416149,12.302303,11.706074,113.412938,2.490142,10.948400,14.873270,11.864452,22.013731,13.804305,0.725653,6.013099,6.319123,89.882765,58.477963,2.881484,6.014868,23.394088,6.499706,8.469745,17.617824,12.294378,12.666975,6.944097,0.134704,8.961780,1.242958,2.943027,21.210805,7.410701,4.437494,2.140666,6.417593,5.027812,14.842479,6.802404,3.812037,8.211003,4.717215,5.174442,14.241535,16.406843,13.722674,5.846928,0.187728,37.452841,5.034674,0.677833,27.415688,9.726098,4.125379,2.557346,7.458893,1.492090,7.572812,18.345046,3.852478,1.650702,5.148883,2.254459,0.212674,0.923411,5.505901,1.817000,5.385269,3.385845,1.104890,1.295312,2.280919,2.259725,1.003451,0

In [14]:
# DISCHARGE.to_clipboard(index=False)

### Pleasee select the runoff type for the analysis (RUNOFF_total, RUNOFF_sub, RUNOFF_sup)

In [8]:
# Select the runoff type 
RUNOFF = RUNOFF_total

### Select reference period from 'year_start' to 'year_end'

Select variable of interest

In [9]:
year_start = 1981
year_end = 2010
# Caudal
SELECTED_REF_DISCHARGE= DISCHARGE[(DISCHARGE['year'] >= year_start) & (DISCHARGE['year'] <= year_end)]
# Escorrentia total
SELECTED_REF_RUNOFF = RUNOFF[(RUNOFF['year'] >= year_start) & (RUNOFF['year'] <= year_end)]
# Precip
SELECTED_REF_PRECIP = PRECIP[(PRECIP['year'] >= year_start) & (PRECIP['year'] <= year_end)]

## Calculate the percentage of anomaly for each basin code level 2 and a specific month and year

In [10]:
#-- OJO: Cambiar fecha
month_analyis = 2
year_analysis = 2026

AVERAGE_PERCENTAGE = pd.DataFrame()
AVERAGE_PERCENTAGE['codigo'] = BASIN_LEVEL3.columns
AVERAGE_PERCENTAGE['discharge'] = np.nan
AVERAGE_PERCENTAGE['runoff'] = np.nan
AVERAGE_PERCENTAGE['precip'] = np.nan



for basin in BASIN_LEVEL3.columns:
    # filter the basin code bas
    filter_col = [col for col in DISCHARGE if col.startswith(str(basin))]
    # Select basins level 3 that belongs to level 2 basin
    # Discharge
    DISCHARGE_FILTER = DISCHARGE[filter_col]
    DISCHARGE_SELECTED = pd.DataFrame()
    DISCHARGE_SELECTED['year'] = DISCHARGE['year']
    DISCHARGE_SELECTED['month'] = DISCHARGE['month']    
    DISCHARGE_SELECTED['discharge'] = DISCHARGE_FILTER.sum(axis=1)
    DISCHARGE_REF = DISCHARGE_SELECTED[(DISCHARGE_SELECTED['year'] >= year_start) & (DISCHARGE_SELECTED['year'] <= year_end)]
    # Runoff
    RUNOFF_FILTER = RUNOFF[filter_col]
    RUNOFF_SELECTED = pd.DataFrame()
    RUNOFF_SELECTED['year'] = RUNOFF['year']
    RUNOFF_SELECTED['month'] = RUNOFF['month']    
    RUNOFF_SELECTED['runoff'] = RUNOFF_FILTER.mean(axis=1)
    RUNOFF_REF = RUNOFF_SELECTED[(RUNOFF_SELECTED['year'] >= year_start) & (RUNOFF_SELECTED['year'] <= year_end)]
    # Precip
    PRECIP_FILTER = PRECIP[filter_col]
    PRECIP_SELECTED = pd.DataFrame()
    PRECIP_SELECTED['year'] = PRECIP['year']
    PRECIP_SELECTED['month'] = PRECIP['month']    
    PRECIP_SELECTED['precip'] = PRECIP_FILTER.mean(axis=1)
    PRECIP_REF = PRECIP_SELECTED[(PRECIP_SELECTED['year'] >= year_start) & (PRECIP_SELECTED['year'] <= year_end)]
    # CALCULATE CLIMATOLOGICAL NORMALS
    # Discharge
    DISCHARGE_AVERAGE = DISCHARGE_REF.groupby(DISCHARGE_REF.month).mean()
    # Runoff
    RUNOFF_AVERAGE = RUNOFF_REF.groupby(RUNOFF_REF.month).mean()
    # Precip
    PRECIP_AVERAGE = PRECIP_REF.groupby(PRECIP_REF.month).mean()
    # HydroSOS Category
    hydroSOS = defineHydroSOScategory(RUNOFF_SELECTED,RUNOFF_AVERAGE,'runoff')
    # Calculate the anomaly
    AVERAGE_PERCENTAGE.loc[AVERAGE_PERCENTAGE.eval('codigo==@basin'),'discharge'] = (DISCHARGE_SELECTED.query('month == @month_analyis & year == @year_analysis')['discharge'].item() - DISCHARGE_AVERAGE.query('month == @month_analyis')['discharge'].item()) / DISCHARGE_AVERAGE.query('month == @month_analyis')['discharge'].item()
    AVERAGE_PERCENTAGE.loc[AVERAGE_PERCENTAGE.eval('codigo==@basin'),'runoff'] = (RUNOFF_SELECTED.query('month == @month_analyis & year == @year_analysis')['runoff'].item()- RUNOFF_AVERAGE.query('month == @month_analyis')['runoff'].item()) / RUNOFF_AVERAGE.query('month == @month_analyis')['runoff'].item()
    AVERAGE_PERCENTAGE.loc[AVERAGE_PERCENTAGE.eval('codigo==@basin'),'precip'] = (PRECIP_SELECTED.query('month == @month_analyis & year == @year_analysis')['precip'].item() - PRECIP_AVERAGE.query('month == @month_analyis')['precip'].item()) / PRECIP_AVERAGE.query('month == @month_analyis')['precip'].item()
    AVERAGE_PERCENTAGE.loc[AVERAGE_PERCENTAGE.eval('codigo==@basin'),'precip_val'] = PRECIP_SELECTED.query('month == @month_analyis & year == @year_analysis')['precip'].item()
    AVERAGE_PERCENTAGE.loc[AVERAGE_PERCENTAGE.eval('codigo==@basin'),'hydroSOS_category'] = hydroSOS.query('month == @month_analyis & year == @year_analysis')['percentile_range'].item()

In [11]:
HTML(AVERAGE_PERCENTAGE.head(10).to_html(index=False))

codigo,discharge,runoff,precip,precip_val,hydroSOS_category
101,-0.842013,-0.843072,-0.284701,109.34,Below normal
102,-0.881729,-0.882653,-0.143959,130.13,Normal range
103,-0.776678,-0.778418,-0.014349,148.94,Normal range
105,-0.368915,-0.372934,0.167489,166.57,Normal range
106,-0.769975,-0.771537,0.179173,169.33,Normal range
107,-0.780141,-0.781865,0.045983,156.01,Normal range
108,-0.730680,-0.732550,0.033499,150.94,Normal range
109,-0.261216,-0.266942,0.485212,203.86,Normal range
110,-0.210598,-0.215590,0.440601,212.81,Normal range
111,-0.102415,-0.109404,0.584279,216.44,Normal range


In [12]:
AVERAGE_PERCENTAGE.to_csv('../qgis_status_outlook/csvtables/modelling_result_codcuenca_3.csv',index=False)
# AVERAGE_PERCENTAGE.to_clipboard(index=False)

## Calculate the percentage of anomaly for each basin code level 2 and a specific year

In [ ]:
#--
year_analysis = 2019

AVERAGE_PERCENTAGE_YEAR = pd.DataFrame()
AVERAGE_PERCENTAGE_YEAR['codigo'] = BASIN_LEVEL2.columns
AVERAGE_PERCENTAGE_YEAR['runoff'] = np.nan
AVERAGE_PERCENTAGE_YEAR['runoff_anomaly'] = np.nan
AVERAGE_PERCENTAGE_YEAR['runoff_mean'] = np.nan



for basin in BASIN_LEVEL3.columns:
    # filter the basin code bas
    filter_col = [col for col in DISCHARGE if col.startswith(str(basin))]
    # Select basins level 3 that belongs to level 2 basin
    # Runoff
    RUNOFF_FILTER = RUNOFF[filter_col]
    RUNOFF_SELECTED = pd.DataFrame()
    RUNOFF_SELECTED['year'] = RUNOFF['year']
    RUNOFF_SELECTED['month'] = RUNOFF['month']    
    RUNOFF_SELECTED['runoff'] = RUNOFF_FILTER.mean(axis=1)
    RUNOFF_REF = RUNOFF_SELECTED[(RUNOFF_SELECTED['year'] >= year_start) & (RUNOFF_SELECTED['year'] <= year_end)]

    sum_avg_runoff_year = RUNOFF_SELECTED.groupby(['year'])['runoff'].sum()
    sum_avg_runoff_year = sum_avg_runoff_year.to_frame()

    # Runoff
    RUNOFF_AVERAGE = RUNOFF_REF.groupby(['year'])['runoff'].sum()
    RUNOFF_AVERAGE = RUNOFF_AVERAGE.to_frame()

    # Calculate the anomaly
    AVERAGE_PERCENTAGE_YEAR.loc[AVERAGE_PERCENTAGE_YEAR.eval('codigo==@basin'),'runoff'] = sum_avg_runoff_year.query('year == @year_analysis')['runoff'].item()
    AVERAGE_PERCENTAGE_YEAR.loc[AVERAGE_PERCENTAGE_YEAR.eval('codigo==@basin'),'runoff_anomaly'] = (sum_avg_runoff_year.query('year == @year_analysis')['runoff'].item() - RUNOFF_AVERAGE['runoff'].mean()) / RUNOFF_AVERAGE['runoff'].mean()
    AVERAGE_PERCENTAGE_YEAR.loc[AVERAGE_PERCENTAGE_YEAR.eval('codigo==@basin'),'runoff_mean'] = RUNOFF_AVERAGE['runoff'].mean()

In [ ]:
HTML(AVERAGE_PERCENTAGE_YEAR.head(6).to_html(index=False))